In [ ]:
import sys
import os

current_dir = os.getcwd()
src_dir = os.path.dirname(current_dir)
sys.path.append(src_dir)

import polars as pl
from pipeline.config import PipelineConfig, DataConfig, ProcessingConfig, StorageConfig, RayConfig, S3Location
from pipeline.pipeline import Pipeline
from data_preprocessing.data_normalization import BMLLNormalizer
from data_preprocessing.data_access import DataAccessFactory

In [ ]:
# 2. Run pipeline
config = PipelineConfig(
    region='us-east-1',
    data=DataConfig(
        raw_data_path='s3://bmlldata',
        start_date='2024-01-02',
        end_date='2024-01-02',
        exchanges=['ARCX'],
        data_types=['trades']
    ),
    processing=ProcessingConfig(
        normalization=BMLLNormalizer()
    ),
    storage=StorageConfig(
        raw_data=S3Location(path='s3://bmlldata'),
        normalized=S3Location(path='s3://orderflowanalysis/intermediate/normalized'),
        features=S3Location(path='s3://orderflowanalysis/intermediate/features'),
        models=S3Location(path='s3://orderflowanalysis/output/models'),
        predictions=S3Location(path='s3://orderflowanalysis/output/predictions'),
        backtest=S3Location(path='s3://orderflowanalysis/output/backtest')
    ),
    ray=RayConfig(runtime_env={"working_dir": src_dir},
                  flat_core_count=5,
                  memory_multiplier=2.0,
                  memory_per_core_gb=4.0,
                  max_retries=5,
                  cpu_buffer=1,
                  file_sort_order="desc")
)

pipeline = Pipeline(config)
results = pipeline.run(slice(10))

In [ ]:
# 3. Verify results
import pandas as pd
res_pd = pd.DataFrame(results)
successful = res_pd.query("message == 'success'").shape[0]
failed = res_pd.query("message != 'success'").shape[0]
print(f"Successful: {successful}, Failed: {failed}")

In [ ]:
# 4. Inspect paths
result = results[0]
print(f"Input:  {result['input_path']}")
print(f"Output: {result['output_path']}")
print(f"Rows:   {result['row_count']:,}")

In [ ]:
# 5. Read output
data_access = DataAccessFactory.create('s3', region='us-east-1')
output_data = data_access.read(result['output_path']).limit(100).collect()
print(f"✓ Read {len(output_data)} records")
output_data.head()

In [ ]:
# 6. Validate schema
normalizer = BMLLNormalizer()
expected_schema = normalizer.get_schema('trades')
missing = set(expected_schema.keys()) - set(output_data.columns)
assert not missing, f"Missing columns: {missing}"
print(f"✓ Schema validated ({len(output_data.columns)} columns)")

In [ ]:
# 7. Compare raw input files with normalized output files
import pandas as pd

raw_files = pipeline._discover_files()
norm_data_access = DataAccessFactory.create('s3', region='us-east-1')
norm_files = config.processing.normalization.discover_files(
    norm_data_access,
    config.storage.normalized.path,
    config.ray.file_sort_order
)

def extract_file_id(path):
    parts = path.split('/')
    if len(parts) >= 6:
        return '/'.join(parts[-6:])
    return path.split('/')[-1]

raw_dict = {extract_file_id(f[0]): {'path': f[0], 'size_gb': f[1]} for f in raw_files}
norm_dict = {extract_file_id(f[0]): {'path': f[0], 'size_gb': f[1]} for f in norm_files}

comparison_data = []
for file_id in set(raw_dict.keys()) & set(norm_dict.keys()):
    comparison_data.append({
        'file_id': file_id,
        'raw_size_gb': raw_dict[file_id]['size_gb'],
        'norm_size_gb': norm_dict[file_id]['size_gb'],
        'size_ratio': norm_dict[file_id]['size_gb'] / raw_dict[file_id]['size_gb'] if raw_dict[file_id]['size_gb'] > 0 else 0,
        'raw_path': raw_dict[file_id]['path'],
        'norm_path': norm_dict[file_id]['path']
    })

comparison_df = pd.DataFrame(comparison_data)
print(f"Avg size ratio: {comparison_df['size_ratio'].mean():.3f}")
display(comparison_df.head(20))

In [ ]:
# 8. Compare dataframes for files with unusual size ratios
unusual = comparison_df[(comparison_df['size_ratio'] < 0.8) | (comparison_df['size_ratio'] > 1.2)].copy()
unusual['data_type'] = unusual['file_id'].str.split('/').str[-3]
trades = unusual[unusual['data_type'] == 'trades'].head(5)
l2 = unusual[unusual['data_type'] == 'level2q'].head(5)

print(f"Found {len(unusual)} files with unusual size ratios")
print(f"Trades: {len(trades)}, L2: {len(l2)}")

for idx, row in pd.concat([trades, l2]).iterrows():
    print(f"\n{'='*80}")
    print(f"File: {row['file_id']}")
    print(f"Size ratio: {row['size_ratio']:.3f}")
    
    raw_df = data_access.read(row['raw_path']).limit(1000).collect()
    norm_df = data_access.read(row['norm_path']).limit(1000).collect()
    
    print(f"Raw shape: {raw_df.shape}, Norm shape: {norm_df.shape}")
    display(raw_df.head(3))
    display(norm_df.head(3))